# Secant and Newton-Raphson method

&nbsp;[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/ExecutableEngineering/blob/main/chapters/root_finding/open_methods/secant_and_newton-raphson.ipynb)

In [ ]:
import numpy as np
import plotly.graph_objects as go

try:
    import executable_engineering as exe
except ImportError:
    %pip install -q executable_engineering
    import executable_engineering as exe

from executable_engineering import RootFinderOpen

## Secant method

Let's reconsider the method of False Position but now we always disregard bracket updating and simply take $c$ as our new guess. Our algorithm is now:

> Take 2 initial guesses: $x^i$
>
> Calculate the next guess:

$$x^{i+1} = {x^i} - f(x^{i}) \frac{x^i-x^{i-1}}{f(x^i)-f(x^{i-1})} $$

> Check if tolerance is met

(Which tolerance?)

In [ ]:
def f(x):
  return x**2 - 2

def secant_method(f, x0, x1, tolerance=1e-6, max_iterations=100):
  print(f"Iteration 0: x = {x1}")
  x_values = [x0, x1]
  for i in range(max_iterations):
    x_new = x1 - f(x1) * (x1 - x0) / (f(x1) - f(x0))
    x_values.append(x_new)
    print(f"Iteration {i+1}: x = {x_new}")
    if abs(f(x_new)) < tolerance:
      return x_new, x_values
    x0 = x1
    x1 = x_new
  return None, x_values

x0 = 1
x1 = 2
root, x_values = secant_method(f, x0, x1)

x = np.linspace(.9, 2.1, 100)
fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=f(x), mode='lines', name='f(x)'))
fig.add_trace(go.Scatter(x=[x0, x1], y=[f(x0), f(x1)], mode='markers', name='Initial guesses'))
import plotly.express as px
colors = px.colors.qualitative.Plotly

for i in range(len(x_values) - 1):
    fig.add_trace(go.Scatter(x=[x_values[i], x_values[i+1]], y=[f(x_values[i]), f(x_values[i+1])], mode='lines', line=dict(dash='dash', color='gray'), showlegend=False))
    color = colors[i % len(colors)]
    fig.add_trace(go.Scatter(x=[x_values[i+1]], y=[0], mode='markers', marker=dict(size=10, color=color), name=f'x_{i+2}'))

if root:
    fig.add_trace(go.Scatter(x=[root], y=[0], mode='markers', marker=dict(color='green', size=10), name='Approximate root'))

fig.update_layout(title='Secant Method', xaxis_title='x', yaxis_title='f(x)')
fig.show()


## The Newton-Raphson method

The Secant method maintains superlinear convergence... but we can do better with a little more information...

Take another look at the fraction in the Secant method update equation:

$$x^{i+1} = {x^i} - f(x^{i}) \frac{x^i-x^{i-1}}{f(x^i)-f(x^{i-1})} $$

This is an (inverse) approximation of $\frac{\partial f}{\partial x}$, the derivative of $f$! Typically, we are able to find this quantity, and the algorithm becomes:

$$ x^{i+1} = {x^i} - \frac{f(x^{i})}{f'(x^i)} $$

or, solving for the increment $\Delta x = x^{i+1}-x^i$ and dropping the indicies,

$$ \begin{align}
\Delta x &= - \frac{f(x)}{f'(x)} \\
f'(x) \Delta x &= - f(x)
\end{align} $$

In [ ]:
def df(x):
  return 2*x

def newton_raphson(f, df, x0, tolerance=1e-6, max_iterations=100):
  x_values = [x0]
  print(f"Iteration 0: x = {x0}")
  for i in range(max_iterations):
    x_new = x0 - f(x0) / df(x0)
    x_values.append(x_new)
    print(f"Iteration {i+1}: x = {x_new}")
    if abs(f(x_new)) < tolerance:
      return x_new, x_values
    x0 = x_new
  return None, x_values

x0 = 1
root, x_values = newton_raphson(f, df, x0)

x = np.linspace(.9, 2.1, 100)
fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=f(x), mode='lines', name='f(x)'))
fig.add_trace(go.Scatter(x=[x0], y=[f(x0)], mode='markers', name='Initial guess'))
import plotly.express as px
colors = px.colors.qualitative.Plotly

for i in range(len(x_values) - 1):
    fig.add_trace(go.Scatter(x=[x_values[i], x_values[i+1]], y=[f(x_values[i]), f(x_values[i+1])], mode='lines', line=dict(dash='dash', color='gray'), showlegend=False))
    color = colors[i % len(colors)]
    fig.add_trace(go.Scatter(x=[x_values[i+1]], y=[0], mode='markers', marker=dict(size=10, color=color), name=f'x_{i+1}'))

if root:
    fig.add_trace(go.Scatter(x=[root], y=[0], mode='markers', marker=dict(color='green', size=10), name='Approximate root'))

fig.update_layout(title='Newton-Raphson Method', xaxis_title='x', yaxis_title='f(x)')
fig.show()


## Interactive Comparison

The Newton-Raphson method has quadratic convergence ($k=2$) *near the root* which is a great result! It does so, however at the cost of calculating the Jacobian and solving a linear system.

<iframe 
    src="https://themintlab.github.io/ENGPHYS_3NM4/Book/assets/Netwon_Secant.html" 
    width="800px" 
    height="750px" 
    frameborder="0"
    allowfullscreen>
</iframe>

In [ ]:
f = lambda x: x**3 - 27
fprime = lambda x: 3 * x**2

x0 = 27.0
x1 = 25.0
tol = 1e-6
max_iter = 20

solver = RootFinderOpen(f=f, fprime=fprime, x0=x0, x1=x1, tol=tol, max_iter=max_iter)
solver.show_toggle_open(interval_ms=750)


## Non-converging example

<iframe 
    src="https://themintlab.github.io/ENGPHYS_3NM4/Book/assets/Newton_Secant_Unconverging_demo.html" 
    width="800px" 
    height="750px" 
    frameborder="0"
    allowfullscreen>
</iframe>

In [ ]:
f = lambda x: x**2 + 2
fprime = lambda x: 2 * x

x0 = 5.0
x1 = 1.0
tol = 1e-6
max_iter = 15

solver = RootFinderOpen(f=f, fprime=fprime, x0=x0, x1=x1, tol=tol, max_iter=max_iter)
solver.show_toggle_open(interval_ms=750)
